# Étape 02 — Classification des communes

Ce notebook montre, PAS À PAS, comment on prédit le cluster d'usage de CHAQUE commune française, même
celles qui n'ont jamais hébergé de capteur : construire le jeu d'entraînement à partir des capteurs
étiquetés (étape 01), entraîner une forêt aléatoire, l'évaluer honnêtement (validation croisée
groupée), puis l'appliquer à toutes les communes.

Trois fichiers, un par rôle (voir leur docstring pour le détail) :

    jeu_de_donnees.py   quel type de commune correspond à quel cluster - construit les données d'entraînement
    entrainement.py      la forêt aléatoire elle-même (hyperparamètres, validation croisée groupée)
    prediction.py          entraîne ET applique le modèle à toutes les communes en une fonction

**Note sur cet environnement de démonstration** : ce notebook a besoin des sorties de
'00_transformation_des_donnees' ('commune_features.parquet', 'sensor_years.parquet') ET de
'01_clustering_des_usages' ('cluster_assignments.parquet') pour produire un vrai résultat - lancez
leur 'pipeline.py' respectif d'abord si elles n'existent pas encore.

## 0. Configuration

Les tables d'entrée viennent des étapes 00 et 01 - pas de configuration propre à ce dossier, seulement
où les lire et où écrire les prédictions/le modèle de CETTE étape.

In [1]:
import sys
from pathlib import Path

ICI = Path.cwd()
sys.path.insert(0, str(ICI))
RACINE = ICI.parents[1]

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

DONNEES_VALIDES = RACINE / "data" / "donnees_valides"
FICHIER_COMMUNES = DONNEES_VALIDES / "commune_features.parquet"
FICHIER_CAPTEURS_ANNEES = DONNEES_VALIDES / "capteurs" / "sensor_years.parquet"
FICHIER_ASSIGNATION_CLUSTERS = DONNEES_VALIDES / "clustering" / "cluster_assignments.parquet"
DOSSIER_SORTIE = DONNEES_VALIDES / "classification"

print("Communes            :", FICHIER_COMMUNES)
print("Capteurs-années      :", FICHIER_CAPTEURS_ANNEES)
print("Assignation clusters :", FICHIER_ASSIGNATION_CLUSTERS)

Communes            : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\commune_features.parquet
Capteurs-années      : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\capteurs\sensor_years.parquet
Assignation clusters : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\clustering\cluster_assignments.parquet


## 1. Charger les données des étapes précédentes

Rien à calculer ici : les trois tables viennent telles quelles des étapes 00 et 01.

In [2]:
communes = pd.read_parquet(FICHIER_COMMUNES)
capteurs = pd.read_parquet(FICHIER_CAPTEURS_ANNEES)
assignation_clusters = pd.read_parquet(FICHIER_ASSIGNATION_CLUSTERS)

print(f"{len(communes):,} communes, {len(capteurs):,} lignes (capteur, année), {len(assignation_clusters):,} capteurs étiquetés")
assignation_clusters.head(5)

34,428 communes, 11,970 lignes (capteur, année), 1,710 capteurs étiquetés


,id_site,annee_reference,modele,cluster
0,200000071,2024,01_classique_hebdomadaire_K4_AkBk,1
1,200000082,2024,01_classique_hebdomadaire_K4_AkBk,2
2,200000083,2024,01_classique_hebdomadaire_K4_AkBk,2
3,200000084,2024,01_classique_hebdomadaire_K4_AkBk,2
4,200000085,2024,01_classique_hebdomadaire_K4_AkBk,0


## 2. Le jeu d'entraînement — ['jeu_de_donnees.py'](jeu_de_donnees.py)

'jeu_entrainement()' fabrique une ligne par CAPTEUR (pas par commune) : ses variables géographiques
(celles de sa commune) et son cluster réel (voir l'étape 01). Une commune à plusieurs capteurs est
automatiquement sous-pondérée (voir la docstring du module) pour ne pas peser plus lourd dans
l'apprentissage.

'effectifs_par_cluster()' et 'clusters_rares()' servent à repérer les clusters trop peu représentés
pour être appris de façon fiable (moins de 10 communes distinctes).

In [3]:
from jeu_de_donnees import clusters_rares, effectifs_par_cluster, jeu_entrainement

labels = assignation_clusters.set_index("id_site")["cluster"]
jeu = jeu_entrainement(communes, capteurs, labels)

print(f"{len(jeu.X):,} capteurs dans {jeu.groupes.nunique():,} communes, {jeu.X.shape[1]} variables")
display(effectifs_par_cluster(jeu))
print("clusters rares (< 10 communes) :", clusters_rares(jeu))

1,589 capteurs dans 1,060 communes, 22 variables


,lignes,communes
cluster,,
0,386,368
1,276,197
2,745,567
3,182,56


clusters rares (< 10 communes) : []


## 3. Entraînement — ['entrainement.py'](entrainement.py)

'ajuster_foret_aleatoire()' ajuste une 'RandomForestClassifier' à classes équilibrées, et l'évalue
honnêtement par validation croisée GROUPÉE par commune ('GroupKFold') - une commune n'est jamais à la
fois dans l'entraînement et la validation d'un même pli, donc le score hors-pli reflète vraiment la
capacité à généraliser à une commune jamais vue.

In [4]:
from entrainement import ajuster_foret_aleatoire

modele = ajuster_foret_aleatoire(jeu, recherche=False)
print("hyperparamètres utilisés :", modele.parametres)
print(f"accuracy équilibrée hors-pli : {modele.accuracy_equilibree_hors_pli:.1%}")
print(f"accuracy hors-pli (brute)    : {modele.accuracy_hors_pli:.1%}")

[classification] forêt aléatoire sur 1589 capteurs : accuracy équilibrée hors-pli 0.5041, accuracy 0.5003
hyperparamètres utilisés : {'n_estimators': 800, 'min_samples_split': 30, 'min_samples_leaf': 8, 'max_features': 0.5, 'max_depth': None}
accuracy équilibrée hors-pli : 50.4%
accuracy hors-pli (brute)    : 50.0%


## 4. Prédire toutes les communes — ['prediction.py'](prediction.py)

'predire_communes()' applique le modèle aux 34 428 communes de France métropolitaine - pas seulement
celles qui ont servi à l'entraînement. La colonne 'extrapolee' distingue les communes dont le cluster
est une observation directe (elles ont fourni un capteur étiqueté) de celles dont c'est une pure
prédiction.

In [5]:
from prediction import predire_communes

observees = set(jeu.groupes.unique())
predictions = predire_communes(modele, communes, observees)

print(f"{len(predictions):,} communes classées, dont {int((predictions['extrapolee'] == 0).sum()):,} observées directement")
predictions["cluster_predit"].value_counts().sort_index()

34,428 communes classées, dont 1,060 observées directement


cluster_predit
0    27176
1     2588
2     4590
3       74
Name: count, dtype: int64

## 5. Écrire les résultats

Comme le ferait 'pipeline.py' : les prédictions en parquet, le modèle en joblib (voir
'entrainement.ModeleClassification.sauvegarder' / '.charger').

In [6]:
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
chemin_predictions = DOSSIER_SORTIE / "commune_clusters.parquet"
predictions.to_parquet(chemin_predictions, index=False)
print("écrit :", chemin_predictions)

écrit : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\classification\commune_clusters.parquet
